In [1]:
# Шаг 1. Импорт библиотек
#
# Логика:
# 1. pandas — для работы с таблицами (чтение CSV, группировки, объединения)
# 2. numpy — для работы с числами и NaN (пропущенными значениями)
# 3. re — для удаления пробелов (включая неразрывные) в числах
# 4. matplotlib.pyplot — для построения графиков
# 5. openpyxl — для форматирования Excel-отчёта (шрифты, границы, заливки)
# 6. tempfile и os — для временного сохранения графика

import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, Border, Side, PatternFill
from openpyxl.utils import get_column_letter
from openpyxl.drawing.image import Image
import tempfile
import os

In [2]:
# Шаг 2. Загрузка исходных данных
#
# Логика:
# 1. df_fin — финансовые данные: по проектам и месяцам суммы отгрузок
# 2. df_prol — данные о пролонгациях: последний месяц реализации проекта и менеджер

df_fin = pd.read_csv(' financial_data.csv')
df_prol = pd.read_csv('prolongations.csv')

In [3]:
# Шаг 3. Функция преобразования строки в число
#
# Логика:
# 1. Если значение не строка (например, уже число или NaN) — возвращаем как есть
# 2. Приводим строку к нижнему регистру и убираем пробелы по краям
# 3. Если значение 'стоп', 'end', 'в ноль' — оставляем строкой (позже обработаем)
# 4. Для чисел: удаляем все пробелы (включая неразрывные \xa0) через re.sub
# 5. Заменяем запятую на точку (десятичный разделитель)
# 6. Преобразуем в float
# 7. Если преобразование не удалось — возвращаем исходную строку

def str_to_float(val):
    # Если это не строка (число, NaN, None) — возвращаем без изменений
    if not isinstance(val, str):
        return val

    # Убираем пробелы по краям и приводим к нижнему регистру для единообразия
    val_clean = val.strip().lower()

    # Служебные значения оставляем строками, они будут обработаны отдельно
    if val_clean in ['стоп', 'end', 'в ноль']:
        return val_clean

    # Пробуем преобразовать в число
    try:
        # re.sub(r'\s+', '', val) — удаляет все пробелы (обычные, неразрывные, табуляции)
        val_no_spaces = re.sub(r'\s+', '', val)
        # Меняем запятую на точку (например, '59 600,00' -> '59600.00')
        val_dot = val_no_spaces.replace(',', '.')
        # Преобразуем в число с плавающей точкой
        return float(val_dot)
    except:
        # Если преобразование не удалось (например, строка 'abc') — возвращаем исходную строку
        return val

In [4]:
# Шаг 4. Приведение названий колонок и месяцев к нижнему регистру
#
# Логика:
# 1. Названия колонок в df_fin (например, 'Ноябрь 2022') приводим к нижнему регистру
# 2. Значения в колонке 'month' в df_prol (например, 'Ноябрь 2022') тоже приводим к нижнему регистру
#    Это нужно для корректного сравнения и поиска

# Переименовываем все колонки в df_fin: 'Ноябрь 2022' -> 'ноябрь 2022'
df_fin.columns = [col.lower() for col in df_fin.columns]

# Переименовываем значения в колонке 'month' в df_prol
df_prol['month'] = df_prol['month'].str.lower()

In [5]:
# Шаг 5. Создание числовых колонок для каждого месяца
#
# Логика:
# 1. Определяем список колонок, которые являются месяцами (все, кроме id, причина дубля, account)
# 2. Для каждого месяца создаём новую колонку с суффиксом '_num'
# 3. Применяем функцию str_to_float к каждой ячейке исходной колонки
#    Например, 'ноябрь 2022' -> 'ноябрь 2022_num' с числами вместо строк

# Список колонок-месяцев (исключаем служебные колонки)
month_cols = [col for col in df_fin.columns if col not in ['id', 'причина дубля', 'account']]

# Для каждого месяца создаём числовую колонку
for col in month_cols:
    df_fin[col + '_num'] = df_fin[col].apply(str_to_float)

In [6]:
# Шаг 6. Исключение проектов, завершившихся досрочно
#
# Логика:
# 1. Проходим по каждой записи в df_prol (каждая запись = один период проекта)
# 2. Для каждого проекта находим его строки в df_fin
# 3. Проверяем значение в последний месяц реализации (из df_prol)
# 4. Если в ячейке стоит 'стоп' или 'end' — проект закрыт досрочно, исключаем эту запись
# 5. Если проект не найден в df_fin — выводим предупреждение и исключаем
# 6. Сохраняем только валидные записи в df_prol_valid

# Список для хранения валидных записей
valid_prolongations = []

# Перебираем каждую строку в df_prol
for idx, row in df_prol.iterrows():
    project_id = row['id']           # ID проекта
    last_month = row['month']        # Последний месяц реализации

    # Находим все строки этого проекта в df_fin
    project_rows = df_fin[df_fin['id'] == project_id]

    # Если проект есть в финансовых данных
    if len(project_rows) > 0:
        # Проверяем, есть ли среди строк 'стоп' или 'end' в последний месяц
        has_stop = False
        for _, fin_row in project_rows.iterrows():
            # Если колонка с нужным месяцем существует
            if last_month in df_fin.columns:
                value = fin_row[last_month]
                # Проверяем, является ли значение 'стоп' или 'end'
                if isinstance(value, str) and value.lower() in ['стоп', 'end']:
                    has_stop = True
                    break

        # Если 'стоп'/'end' не найдены — запись валидна
        if not has_stop:
            valid_prolongations.append(row)
    else:
        # Проект есть в df_prol, но нет в df_fin — аномалия, исключаем
        print(f"ВНИМАНИЕ: Проект {project_id} есть в prolongations, но не найден в financial_data. Исключаем.")

# Создаём новую таблицу только с валидными записями
df_prol_valid = pd.DataFrame(valid_prolongations)

# Выводим статистику
print(f"Было записей: {len(df_prol)}")
print(f"Осталось записей: {len(df_prol_valid)}")
print(f"Исключено записей: {len(df_prol) - len(df_prol_valid)}")

Было записей: 477
Осталось записей: 439
Исключено записей: 38


In [7]:
# Шаг 7. Обработка значения "в ноль"
#
# Логика:
# 1. Проходим по каждой ячейке в df_fin (исходные данные, ещё до суммирования дублей)
# 2. Если в ячейке написано 'в ноль', проверяем другие строки с тем же id за этот месяц
# 3. Если есть хотя бы одна числовая строка (не 'в ноль', не 'стоп', не 'end', не пусто) —
#    значит сумма за месяц будет ненулевой, заменяем 'в ноль' на 0
# 4. Если числовых строк нет — заменяем 'в ноль' на значение из предыдущего месяца
# 5. Если это первый месяц (например, ноябрь 2022) — предыдущего нет, заменяем на 0

# Проходим по каждой строке df_fin
for idx, row in df_fin.iterrows():
    # Проходим по каждой колонке-месяцу
    for col in month_cols:
        val = row[col]

        # Проверяем, является ли значение строкой 'в ноль'
        if isinstance(val, str) and val.strip().lower() == 'в ноль':

            # Ищем другие строки с тем же id за этот же месяц
            other_rows = df_fin[(df_fin['id'] == row['id']) & (df_fin.index != idx)]
            has_number = False

            # Проверяем каждую другую строку на наличие числа
            for _, other_row in other_rows.iterrows():
                other_val = other_row[col]

                # Если значение — число (int или float) → нашли числовой дубль
                if isinstance(other_val, (int, float)):
                    has_number = True
                    break

                # Если значение — строка, проверяем не является ли она служебной
                elif isinstance(other_val, str):
                    other_clean = other_val.strip().lower()
                    # Если это не 'в ноль', не 'стоп', не 'end', не 'nan' — значит это число в виде строки
                    if other_clean not in ['в ноль', 'стоп', 'end', 'nan']:
                        has_number = True
                        break

            # Если есть числовой дубль → заменяем 'в ноль' на 0
            if has_number:
                df_fin.at[idx, col] = 0
            else:
                # Нет числовых дублей → заменяем на значение из предыдущего месяца
                col_idx = month_cols.index(col)

                # Если это не первый месяц
                if col_idx > 0:
                    prev_col = month_cols[col_idx - 1]   # Название предыдущего месяца
                    prev_val = row[prev_col]              # Значение в предыдущем месяце

                    # Если предыдущее значение тоже 'в ноль' — ставим 0, иначе берём предыдущее
                    if isinstance(prev_val, str) and prev_val.strip().lower() == 'в ноль':
                        df_fin.at[idx, col] = 0
                    else:
                        df_fin.at[idx, col] = prev_val
                else:
                    # Это первый месяц в данных (ноябрь 2022) — предыдущего нет, ставим 0
                    df_fin.at[idx, col] = 0

print("Обработка 'в ноль' завершена")

Обработка 'в ноль' завершена


In [8]:
# Шаг 8. Суммирование дублей (несколько строк на один проект и месяц)
#
# Логика:
# 1. Создаём копию таблицы, чтобы не изменять оригинал
# 2. В числовых колонках (с суффиксом '_num') заменяем все строки на 0
#    (это безопасно, так как все значимые строки мы уже обработали на шаге 7)
# 3. Группируем строки по id и суммируем значения по всем числовым колонкам
#    Если у проекта было несколько строк за один месяц (например, первая и вторая часть оплаты),
#    они сложатся в одну сумму

# Создаём копию таблицы
df_fin_for_sum = df_fin.copy()

# Получаем список всех числовых колонок-месяцев (с суффиксом '_num')
num_month_cols = [col for col in df_fin.columns if col.endswith('_num')]

# Заменяем все строки на 0 в этих колонках (чтобы при суммировании не было ошибок)
for col in num_month_cols:
    df_fin_for_sum[col] = df_fin_for_sum[col].apply(
        lambda x: 0 if isinstance(x, str) else x
    )

# Группируем по id и суммируем все числовые колонки
df_fin_summed = df_fin_for_sum.groupby('id')[num_month_cols].sum().reset_index()

# Выводим статистику
print(f"Было строк в df_fin: {len(df_fin)}")
print(f"Стало строк после суммирования дублей: {len(df_fin_summed)}")

Было строк в df_fin: 451
Стало строк после суммирования дублей: 314


In [9]:
# Шаг 9. Функция преобразования названия месяца в числовой код
#
# Логика:
# 1. Сопоставляем название месяца с его номером (январь = 1, февраль = 2, ...)
# 2. Вычисляем числовой код: год * 12 + номер месяца
# 3. Например, 'январь 2023' -> 2023*12 + 1 = 24277
# 4. Это позволяет легко находить соседние месяцы: предыдущий = код - 1, следующий = код + 1

def month_name_to_num(month_name):
    # Словарь для перевода названия месяца в номер
    month_map = {
        'январь': 1, 'февраль': 2, 'март': 3, 'апрель': 4,
        'май': 5, 'июнь': 6, 'июль': 7, 'август': 8,
        'сентябрь': 9, 'октябрь': 10, 'ноябрь': 11, 'декабрь': 12
    }
    # Разделяем строку 'январь 2023' на ['январь', '2023']
    parts = month_name.split()
    month_str = parts[0]   # название месяца
    year = int(parts[1])   # год
    # Возвращаем числовой код
    return year * 12 + month_map[month_str]

In [10]:
# Шаг 10. Преобразование из широкого формата в длинный
#
# Логика:
# 1. Широкий формат: один проект — одна строка, каждый месяц — отдельная колонка
# 2. Длинный формат: каждый месяц — отдельная строка (удобно для фильтрации)
# 3. Собираем все числовые колонки-месяцы (с суффиксом '_num')
# 4. Используем melt для преобразования
# 5. Убираем суффикс '_num' из названий месяцев
# 6. Добавляем числовой код месяца для сортировки
# 7. Сортируем по проекту и месяцу

# Собираем колонки, которые заканчиваются на '_num'
month_cols_num = [col for col in df_fin_summed.columns if col.endswith('_num')]

# Превращаем широкую таблицу в длинную
df_fin_long = df_fin_summed.melt(
    id_vars=['id'],               # id проекта остаётся колонкой
    value_vars=month_cols_num,    # колонки-месяцы превращаются в строки
    var_name='month',             # названия месяцев попадают в колонку 'month'
    value_name='amount'           # суммы попадают в колонку 'amount'
)

# Убираем суффикс '_num' из названий месяцев (было 'ноябрь 2022_num' -> 'ноябрь 2022')
df_fin_long['month'] = df_fin_long['month'].str.replace('_num', '')

# Добавляем числовой код месяца для сортировки
df_fin_long['month_num'] = df_fin_long['month'].apply(month_name_to_num)

# Сортируем по id проекта и по месяцу
df_fin_long = df_fin_long.sort_values(['id', 'month_num'])

print("Длинная таблица готова")

Длинная таблица готова


In [11]:
# Шаг 11. Добавление информации о последнем месяце проекта и менеджере
#
# Логика:
# 1. Добавляем числовой код месяца в df_prol_valid
# 2. Для каждого проекта собираем список всех его последних месяцев
# 3. Для каждого месяца отгрузки находим ближайший последний месяц, который >= этого месяца
#    (отгрузка относится к тому периоду, который ещё не завершился или завершился в этом же месяце)
# 4. Присоединяем информацию о менеджере и названии последнего месяца

# Добавляем числовой код месяца в df_prol_valid
df_prol_valid['month_num'] = df_prol_valid['month'].apply(month_name_to_num)

# Сортируем для корректного поиска
df_prol_valid_sorted = df_prol_valid.sort_values(['id', 'month_num'])

# Группируем: для каждого проекта получаем список всех его последних месяцев
last_months_by_id = df_prol_valid_sorted.groupby('id')['month_num'].apply(list).to_dict()

# Функция для поиска правильного последнего месяца для каждой отгрузки
def find_last_month(row, last_months_dict):
    month_num = row['month_num']           # Месяц отгрузки
    project_id = row['id']                 # ID проекта
    last_months = last_months_dict.get(project_id, [])  # Список последних месяцев проекта

    # Ищем ближайший последний месяц, который >= месяца отгрузки
    for last_num in sorted(last_months):
        if last_num >= month_num:
            return last_num

    # Если не нашли (отгрузка после всех последних месяцев) — возвращаем последний
    return last_months[-1] if last_months else None

# Добавляем временную колонку с подобранным последним месяцем
df_fin_long['matched_last_month_num'] = df_fin_long.apply(
    lambda row: find_last_month(row, last_months_by_id), axis=1
)

# Присоединяем информацию из df_prol_valid (название месяца, менеджер)
df_fin_long = df_fin_long.merge(
    df_prol_valid[['id', 'month', 'month_num', 'AM']],
    left_on=['id', 'matched_last_month_num'],
    right_on=['id', 'month_num'],
    how='inner',
    suffixes=('', '_last')
)

# Переименовываем колонки для ясности
df_fin_long.rename(columns={
    'month_last': 'last_month_name',   # название последнего месяца проекта
    'month_num_last': 'last_month_num', # числовой код последнего месяца
    'AM_last': 'manager'                # менеджер проекта
}, inplace=True)

# Удаляем временную колонку
df_fin_long.drop(columns=['matched_last_month_num'], inplace=True)

print("Информация о последнем месяце добавлена")

Информация о последнем месяце добавлена


In [12]:
# Шаг 12. Функция расчёта коэффициентов пролонгации для одного месяца
#
# Логика:
# 1. Для целевого месяца определяем предыдущий месяц (target - 1) и месяц 2 месяца назад (target - 2)
# 2. Коэффициент 1 (пролонгация в первый месяц):
#    - Берём проекты, у которых последний месяц = предыдущему месяцу
#    - Знаменатель: сумма отгрузок этих проектов за предыдущий месяц
#    - Числитель: сумма отгрузок этих проектов за целевой месяц
#    - Коэффициент = числитель / знаменатель
# 3. Коэффициент 2 (пролонгация во второй месяц):
#    - Берём проекты, у которых последний месяц = месяцу 2 месяца назад
#    - Исключаем те, у которых есть отгрузка в предыдущем месяце (они уже пролонгировались)
#    - Знаменатель: сумма отгрузок оставшихся проектов за месяц 2 месяца назад
#    - Числитель: сумма отгрузок этих проектов за целевой месяц
#    - Коэффициент = числитель / знаменатель

def calc_coefficients_for_month(df, target_month_num):
    # Определяем соседние месяцы
    prev_month_num = target_month_num - 1       # предыдущий месяц
    two_months_ago = target_month_num - 2       # месяц 2 месяца назад

    # === КОЭФФИЦИЕНТ 1 ===
    # Проекты, завершившиеся в предыдущем месяце
    projects_prev = df[df['last_month_num'] == prev_month_num].copy()

    if len(projects_prev) > 0:
        # Знаменатель: сумма отгрузок за предыдущий месяц
        denominator_1 = projects_prev[projects_prev['month_num'] == prev_month_num]['amount'].sum()
        # Числитель: сумма отгрузок за целевой месяц
        numerator_1 = projects_prev[projects_prev['month_num'] == target_month_num]['amount'].sum()
        # Коэффициент
        coeff_1 = numerator_1 / denominator_1 if denominator_1 > 0 else 0
        # Количество проектов для статистики
        projects_count_1 = projects_prev['id'].nunique()
    else:
        coeff_1 = 0
        denominator_1 = 0
        numerator_1 = 0
        projects_count_1 = 0

    # === КОЭФФИЦИЕНТ 2 ===
    # Проекты, завершившиеся 2 месяца назад
    projects_two = df[df['last_month_num'] == two_months_ago].copy()

    if len(projects_two) > 0:
        # Исключаем проекты, у которых есть отгрузка в предыдущем месяце
        projects_without_first = []
        for project_id in projects_two['id'].unique():
            project_data = projects_two[projects_two['id'] == project_id]
            # Проверяем, есть ли отгрузка в предыдущем месяце
            has_prev = (project_data['month_num'] == prev_month_num).any()
            if not has_prev:
                projects_without_first.append(project_id)

        # Оставляем только проекты без отгрузки в предыдущем месяце
        projects_filtered = projects_two[projects_two['id'].isin(projects_without_first)]

        # Знаменатель: сумма отгрузок за месяц 2 месяца назад
        denominator_2 = projects_filtered[projects_filtered['month_num'] == two_months_ago]['amount'].sum()
        # Числитель: сумма отгрузок за целевой месяц
        numerator_2 = projects_filtered[projects_filtered['month_num'] == target_month_num]['amount'].sum()
        # Коэффициент
        coeff_2 = numerator_2 / denominator_2 if denominator_2 > 0 else 0
        # Количество проектов для статистики
        projects_count_2 = projects_filtered['id'].nunique()
    else:
        coeff_2 = 0
        denominator_2 = 0
        numerator_2 = 0
        projects_count_2 = 0

    # Возвращаем результат в виде словаря
    return {
        'month_num': target_month_num,
        'coeff_1': coeff_1,
        'coeff_2': coeff_2,
        'denominator_1': denominator_1,
        'numerator_1': numerator_1,
        'denominator_2': denominator_2,
        'numerator_2': numerator_2,
        'projects_count_1': projects_count_1,
        'projects_count_2': projects_count_2
    }

In [13]:
# Шаг 13. Расчёт коэффициентов для всего отдела за 2023 год
#
# Логика:
# 1. Определяем список всех месяцев 2023 года с их числовыми кодами
# 2. Для каждого месяца вызываем функцию calc_coefficients_for_month
# 3. Добавляем название месяца в результат
# 4. Собираем все результаты в DataFrame

# Список месяцев 2023 года (название, числовой код)
months_2023 = [
    ('январь 2023', 24277),
    ('февраль 2023', 24278),
    ('март 2023', 24279),
    ('апрель 2023', 24280),
    ('май 2023', 24281),
    ('июнь 2023', 24282),
    ('июль 2023', 24283),
    ('август 2023', 24284),
    ('сентябрь 2023', 24285),
    ('октябрь 2023', 24286),
    ('ноябрь 2023', 24287),
    ('декабрь 2023', 24288)
]

# Список для хранения результатов
dept_results = []

# Перебираем все месяцы
for month_name, month_num in months_2023:
    # Вызываем функцию расчёта
    result = calc_coefficients_for_month(df_fin_long, month_num)
    # Добавляем название месяца
    result['month_name'] = month_name
    dept_results.append(result)

# Превращаем список в DataFrame
df_dept = pd.DataFrame(dept_results)

# Переставляем колонки в удобном порядке
df_dept = df_dept[['month_name', 'month_num', 'coeff_1', 'coeff_2',
                   'denominator_1', 'numerator_1', 'denominator_2', 'numerator_2',
                   'projects_count_1', 'projects_count_2']]

# Выводим результат
print("Коэффициенты для отдела за 2023 год:")
print(df_dept.to_string())

Коэффициенты для отдела за 2023 год:
       month_name  month_num   coeff_1  coeff_2  denominator_1  numerator_1  denominator_2  numerator_2  projects_count_1  projects_count_2
0     январь 2023      24277  0.063297      0.0     5986766.86    378941.67     1433647.00          0.0                54                11
1    февраль 2023      24278  0.318215      0.0     2549075.50    811155.00     3365393.39          0.0                20                24
2       март 2023      24279  0.046051      0.0     1716452.77     79045.00     1311255.00          0.0                26                10
3     апрель 2023      24280  0.019398      0.0     3377742.20     65520.00     1063692.77          0.0                31                13
4        май 2023      24281  0.115816      0.0     3647436.50    422430.00     2069484.20          0.0                25                16
5       июнь 2023      24282  0.000000      0.0     1274625.53         0.00     2252556.50          0.0                21  

In [14]:
# Шаг 14. Расчёт коэффициентов по каждому менеджеру за 2023 год
#
# Логика:
# 1. Получаем список уникальных менеджеров из данных
# 2. Для каждого менеджера фильтруем df_fin_long по колонке 'AM'
# 3. Для каждого месяца 2023 года вызываем функцию calc_coefficients_for_month
# 4. Добавляем название месяца и имя менеджера в результат
# 5. Собираем все результаты в общий DataFrame

# Получаем список всех менеджеров (колонка 'AM')
managers = df_fin_long['AM'].unique()
print(f"Найдено менеджеров: {len(managers)}")

# Функция для расчёта по одному менеджеру
def calc_for_manager(df, manager_name, months_list):
    results = []
    # Фильтруем данные только для этого менеджера
    df_manager = df[df['AM'] == manager_name].copy()

    # Перебираем все месяцы
    for month_name, month_num in months_list:
        result = calc_coefficients_for_month(df_manager, month_num)
        result['month_name'] = month_name
        result['manager'] = manager_name
        results.append(result)

    return pd.DataFrame(results)

# Список для хранения результатов по всем менеджерам
all_manager_results = []

# Перебираем всех менеджеров
for manager_name in managers:
    print(f"Обработка: {manager_name}")
    df_mgr = calc_for_manager(df_fin_long, manager_name, months_2023)
    all_manager_results.append(df_mgr)

# Объединяем все результаты в один DataFrame
df_managers = pd.concat(all_manager_results, ignore_index=True)

print("Расчёт по менеджерам завершён")

Найдено менеджеров: 10
Обработка: Иванова Мария Сергеевна
Обработка: Васильев Артем Александрович
Обработка: Попова Екатерина Николаевна
Обработка: Смирнова Ольга Владимировна
Обработка: Соколова Анастасия Викторовна
Обработка: Кузнецов Михаил Иванович
Обработка: Михайлов Андрей Сергеевич
Обработка: Федорова Марина Васильевна
Обработка: без А/М
Обработка: Петрова Анна Дмитриевна
Расчёт по менеджерам завершён


In [15]:
# Шаг 15. Формирование Excel-отчёта (подготовка данных)
#
# Логика:
# 1. Переводим коэффициенты из долей (0.1158) в проценты (11.6%)
# 2. Подготавливаем данные для листа "Весь отдел"
# 3. Подготавливаем данные для листа "Менеджеры за год"
# 4. Подготавливаем сводные таблицы для листа "Менеджеры по месяцам"

# Порядок месяцев для отчёта (январь → декабрь)
month_order = ['январь 2023', 'февраль 2023', 'март 2023', 'апрель 2023', 'май 2023', 'июнь 2023',
               'июль 2023', 'август 2023', 'сентябрь 2023', 'октябрь 2023', 'ноябрь 2023', 'декабрь 2023']

# Функция перевода коэффициента в проценты (округляем до 1 знака)
def to_percent(val):
    if isinstance(val, (int, float)) and not pd.isna(val):
        return round(val * 100, 1)
    return 0

# === ЛИСТ "Весь отдел" ===
df_dept_report = df_dept[['month_name', 'denominator_1', 'numerator_1', 'coeff_1',
                          'denominator_2', 'numerator_2', 'coeff_2']].copy()
df_dept_report.columns = ['Месяц', 'к пролонгации', 'пролонгировано', 'Коэффициент_1',
                          'к пролонгации_2', 'пролонгировано_2', 'Коэффициент_2']
df_dept_report['Коэффициент_1'] = df_dept_report['Коэффициент_1'].apply(to_percent)
df_dept_report['Коэффициент_2'] = df_dept_report['Коэффициент_2'].apply(to_percent)

# Переименовываем для вывода в Excel
df_dept_for_excel = df_dept_report[['Месяц', 'к пролонгации', 'пролонгировано', 'Коэффициент_1',
                                    'к пролонгации_2', 'пролонгировано_2', 'Коэффициент_2']].copy()
df_dept_for_excel.columns = ['Месяц', 'к пролонгации', 'пролонгировано', 'Коэффициент, %',
                             'к пролонгации', 'пролонгировано', 'Коэффициент, %']

# === ЛИСТ "Менеджеры за год" ===
# Агрегируем данные по менеджерам за весь год
df_managers_yearly = df_managers.groupby('manager').agg({
    'denominator_1': 'sum',
    'numerator_1': 'sum',
    'denominator_2': 'sum',
    'numerator_2': 'sum'
}).reset_index()

# Рассчитываем годовые коэффициенты
df_managers_yearly['coeff_1'] = df_managers_yearly['numerator_1'] / df_managers_yearly['denominator_1']
df_managers_yearly['coeff_2'] = df_managers_yearly['numerator_2'] / df_managers_yearly['denominator_2']
df_managers_yearly = df_managers_yearly.fillna(0)

df_managers_yearly_report = df_managers_yearly[['manager', 'denominator_1', 'numerator_1', 'coeff_1',
                                                'denominator_2', 'numerator_2', 'coeff_2']].copy()
df_managers_yearly_report.columns = ['Менеджер', 'к пролонгации', 'пролонгировано', 'Коэффициент_1',
                                     'к пролонгации_2', 'пролонгировано_2', 'Коэффициент_2']
df_managers_yearly_report['Коэффициент_1'] = df_managers_yearly_report['Коэффициент_1'].apply(to_percent)
df_managers_yearly_report['Коэффициент_2'] = df_managers_yearly_report['Коэффициент_2'].apply(to_percent)

df_managers_yearly_for_excel = df_managers_yearly_report[['Менеджер', 'к пролонгации', 'пролонгировано', 'Коэффициент_1',
                                                          'к пролонгации_2', 'пролонгировано_2', 'Коэффициент_2']].copy()
df_managers_yearly_for_excel.columns = ['Менеджер', 'к пролонгации', 'пролонгировано', 'Коэффициент, %',
                                        'к пролонгации', 'пролонгировано', 'Коэффициент, %']

# === ЛИСТ "Менеджеры по месяцам" ===
# Сводная таблица для коэффициента 1
pivot_1 = df_managers.pivot_table(index='manager', columns='month_name', values='coeff_1').reset_index()
pivot_1 = pivot_1[['manager'] + month_order]
pivot_1.columns.name = None
for month in month_order:
    pivot_1[month] = pivot_1[month].apply(to_percent)

# Сводная таблица для коэффициента 2
pivot_2 = df_managers.pivot_table(index='manager', columns='month_name', values='coeff_2').reset_index()
pivot_2 = pivot_2[['manager'] + month_order]
pivot_2.columns.name = None
for month in month_order:
    pivot_2[month] = pivot_2[month].apply(to_percent)

In [16]:
# Шаг 16. Формирование Excel-отчёта (запись и стили)
#
# Логика:
# 1. Создаём новый Excel-файл
# 2. Определяем стили: шрифт Arial 12, границы, выравнивание по центру, цвета заливки
# 3. Заполняем лист "Весь отдел" с объединением ячеек и форматированием
# 4. Заполняем лист "Менеджеры за год"
# 5. Заполняем лист "Менеджеры по месяцам" с двумя блоками (коэффициент 1 и 2)
# 6. Создаём график и добавляем на отдельный лист

# Создаём новый Workbook
wb = Workbook()

# Удаляем стандартный лист "Sheet", он не нужен
if 'Sheet' in wb.sheetnames:
    del wb['Sheet']

# === СТИЛИ ===
# Шрифт для заголовков: Arial 12 жирный
font_header = Font(name='Arial', size=12, bold=True)
# Шрифт для данных: Arial 12 обычный
font_normal = Font(name='Arial', size=12, bold=False)
# Выравнивание: по центру по горизонтали и вертикали
alignment_center = Alignment(horizontal='center', vertical='center')
# Границы: тонкие линии со всех сторон
border = Border(
    left=Side(style='thin'),
    right=Side(style='thin'),
    top=Side(style='thin'),
    bottom=Side(style='thin')
)
# Заливка для заголовков: #fee698
fill_header = PatternFill(start_color='fee698', end_color='fee698', fill_type='solid')
# Заливка для "Коэффициент 1" и "Коэффициент 2": #ffd867
fill_coeff = PatternFill(start_color='ffd867', end_color='ffd867', fill_type='solid')

# === ЛИСТ "Весь отдел" ===
ws = wb.create_sheet('Весь отдел')

# Заполняем значения заголовков
ws['A1'] = 'Месяц'
ws['B1'] = 'Пролонгации в первый месяц'
ws['E1'] = 'Пролонгации через месяц'
ws['A2'] = 'Месяц'
ws['B2'] = 'к пролонгации'
ws['C2'] = 'пролонгировано'
ws['D2'] = 'Коэффициент, %'
ws['E2'] = 'к пролонгации'
ws['F2'] = 'пролонгировано'
ws['G2'] = 'Коэффициент, %'

# Объединяем ячейки согласно шаблону
ws.merge_cells('A1:A2')
ws.merge_cells('B1:D1')
ws.merge_cells('E1:G1')

# Применяем стили к заголовкам (строки 1 и 2, колонки A-G)
for row in [1, 2]:
    for col in range(1, 8):
        cell = ws.cell(row=row, column=col)
        cell.fill = fill_header
        cell.font = font_header
        cell.alignment = alignment_center
        cell.border = border

# Заполняем данными (начиная с 3 строки)
for row_idx, row_data in enumerate(df_dept_for_excel.values.tolist(), start=3):
    for col_idx, val in enumerate(row_data, start=1):
        cell = ws.cell(row=row_idx, column=col_idx)
        cell.value = val
        cell.font = font_normal
        cell.alignment = alignment_center
        cell.border = border

# Устанавливаем ширину столбцов
ws.column_dimensions['A'].width = 18
for col in range(2, 8):
    ws.column_dimensions[get_column_letter(col)].width = 18

# === ЛИСТ "Менеджеры за год" ===
ws2 = wb.create_sheet('Менеджеры за год')

# Заполняем заголовки
ws2['A1'] = 'Менеджер'
ws2['B1'] = 'Пролонгации в первый месяц'
ws2['E1'] = 'Пролонгации через месяц'
ws2['A2'] = 'Менеджер'
ws2['B2'] = 'к пролонгации'
ws2['C2'] = 'пролонгировано'
ws2['D2'] = 'Коэффициент, %'
ws2['E2'] = 'к пролонгации'
ws2['F2'] = 'пролонгировано'
ws2['G2'] = 'Коэффициент, %'

# Объединяем ячейки
ws2.merge_cells('A1:A2')
ws2.merge_cells('B1:D1')
ws2.merge_cells('E1:G1')

# Применяем стили к заголовкам
for row in [1, 2]:
    for col in range(1, 8):
        cell = ws2.cell(row=row, column=col)
        cell.fill = fill_header
        cell.font = font_header
        cell.alignment = alignment_center
        cell.border = border

# Заполняем данными
for row_idx, row_data in enumerate(df_managers_yearly_for_excel.values.tolist(), start=3):
    for col_idx, val in enumerate(row_data, start=1):
        cell = ws2.cell(row=row_idx, column=col_idx)
        cell.value = val
        cell.font = font_normal
        cell.alignment = alignment_center
        cell.border = border

# Устанавливаем ширину столбцов
ws2.column_dimensions['A'].width = 32
for col in range(2, 8):
    ws2.column_dimensions[get_column_letter(col)].width = 18

# === ЛИСТ "Менеджеры по месяцам" ===
ws3 = wb.create_sheet('Менеджеры по месяцам')

# ----- КОЭФФИЦИЕНТ 1 -----
ws3['A1'] = 'Коэффициент 1'
ws3.merge_cells('A1:B1')
cell = ws3['A1']
cell.fill = fill_coeff
cell.font = font_header
cell.alignment = alignment_center
cell.border = border

# Ячейки A2:A3: 'Менеджер' (объединены)
ws3['A2'] = 'Менеджер'
ws3.merge_cells('A2:A3')
cell = ws3['A2']
cell.fill = fill_header
cell.font = font_header
cell.alignment = alignment_center
cell.border = border

# Строка 2, колонки B-M: объединённая ячейка с текстом 'Месяц'
ws3.merge_cells(start_row=2, start_column=2, end_row=2, end_column=2 + len(month_order) - 1)
cell = ws3.cell(row=2, column=2)
cell.value = 'Месяц'
cell.fill = fill_header
cell.font = font_header
cell.alignment = alignment_center
cell.border = border

# Строка 3, колонки B-M: названия месяцев
for i, month in enumerate(month_order, start=2):
    cell = ws3.cell(row=3, column=i)
    cell.value = month
    cell.fill = fill_header
    cell.font = font_header
    cell.alignment = alignment_center
    cell.border = border

# Заливка для A2:M3
for row in [2, 3]:
    for col in range(1, 2 + len(month_order)):
        cell = ws3.cell(row=row, column=col)
        cell.fill = fill_header
        cell.font = font_header
        cell.alignment = alignment_center
        cell.border = border

# Данные коэффициента 1 (начиная с 4 строки)
current_row = 4
for _, row in pivot_1.iterrows():
    # Колонка A: имя менеджера
    cell = ws3.cell(row=current_row, column=1)
    cell.value = row['manager']
    cell.font = font_normal
    cell.alignment = alignment_center
    cell.border = border

    # Колонки B-M: значения по месяцам
    for i, month in enumerate(month_order, start=2):
        val = row[month] if month in row.index else 0
        cell = ws3.cell(row=current_row, column=i)
        cell.value = val
        cell.font = font_normal
        cell.alignment = alignment_center
        cell.border = border

    current_row += 1

# Пустая строка между коэффициентом 1 и коэффициентом 2
current_row += 1

# ----- КОЭФФИЦИЕНТ 2 -----
ws3.cell(row=current_row, column=1).value = 'Коэффициент 2'
ws3.merge_cells(start_row=current_row, start_column=1, end_row=current_row, end_column=2)
cell = ws3.cell(row=current_row, column=1)
cell.fill = fill_coeff
cell.font = font_header
cell.alignment = alignment_center
cell.border = border

current_row += 1

# Ячейки A(строка):A(строка+1): 'Менеджер' (объединены)
ws3.cell(row=current_row, column=1).value = 'Менеджер'
ws3.merge_cells(start_row=current_row, start_column=1, end_row=current_row+1, end_column=1)
cell = ws3.cell(row=current_row, column=1)
cell.fill = fill_header
cell.font = font_header
cell.alignment = alignment_center
cell.border = border

# Строка current_row, колонки B-M: объединённая ячейка с текстом 'Месяц'
ws3.merge_cells(start_row=current_row, start_column=2, end_row=current_row, end_column=2 + len(month_order) - 1)
cell = ws3.cell(row=current_row, column=2)
cell.value = 'Месяц'
cell.fill = fill_header
cell.font = font_header
cell.alignment = alignment_center
cell.border = border

current_row += 1

# Строка current_row, колонки B-M: названия месяцев
for i, month in enumerate(month_order, start=2):
    cell = ws3.cell(row=current_row, column=i)
    cell.value = month
    cell.fill = fill_header
    cell.font = font_header
    cell.alignment = alignment_center
    cell.border = border

# Заливка заголовков коэффициента 2
for col in range(1, 2 + len(month_order)):
    for r in [current_row - 1, current_row]:
        cell = ws3.cell(row=r, column=col)
        cell.fill = fill_header
        cell.font = font_header
        cell.alignment = alignment_center
        cell.border = border

# Данные коэффициента 2
current_row += 1
for _, row in pivot_2.iterrows():
    cell = ws3.cell(row=current_row, column=1)
    cell.value = row['manager']
    cell.font = font_normal
    cell.alignment = alignment_center
    cell.border = border

    for i, month in enumerate(month_order, start=2):
        val = row[month] if month in row.index else 0
        cell = ws3.cell(row=current_row, column=i)
        cell.value = val
        cell.font = font_normal
        cell.alignment = alignment_center
        cell.border = border

    current_row += 1

# Ширина столбцов на листе 3
ws3.column_dimensions['A'].width = 32
for i in range(2, 2 + len(month_order)):
    ws3.column_dimensions[get_column_letter(i)].width = 14

# === ЛИСТ С ГРАФИКОМ ===
# Сохраняем график во временный файл
chart_path = 'coeff_dynamics.png'

# Создаём фигуру и оси
fig, ax = plt.subplots(figsize=(14, 8))

# Выбираем топ-5 менеджеров по среднему коэффициенту 1
managers_avg = df_managers.groupby('manager')['coeff_1'].mean().sort_values(ascending=False)
top_managers = managers_avg.head(5).index.tolist()
# Добавляем отдел для сравнения
managers_to_plot = top_managers + ['ОТДЕЛ']

# Цветовая палитра
colors = plt.cm.tab10(range(len(managers_to_plot)))

# Строим линии для каждого менеджера
for i, manager in enumerate(managers_to_plot):
    if manager == 'ОТДЕЛ':
        # Данные отдела из df_dept
        values = [df_dept[df_dept['month_name'] == month]['coeff_1'].values[0] * 100 for month in month_order]
        ax.plot(month_order, values, marker='o', linewidth=3, label=manager, color='black', linestyle='--')
    else:
        # Данные менеджера из df_managers
        mgr_data = df_managers[df_managers['manager'] == manager]
        values = [mgr_data[mgr_data['month_name'] == month]['coeff_1'].values[0] * 100 if len(mgr_data[mgr_data['month_name'] == month]) > 0 else 0 for month in month_order]
        ax.plot(month_order, values, marker='o', linewidth=2, label=manager, color=colors[i], alpha=0.8)

# Настройки графика
ax.set_xlabel('Месяц', fontsize=12)
ax.set_ylabel('Коэффициент пролонгации (%)', fontsize=12)
ax.set_title('Динамика коэффициента пролонгации (топ-5 менеджеров и отдел)', fontsize=14)
ax.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=10)
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()

# Сохраняем график
plt.savefig(chart_path, dpi=150, bbox_inches='tight')
plt.close()

# Добавляем лист с графиком
ws_chart = wb.create_sheet('График')
img = Image(chart_path)
img.width = 800
img.height = 500
ws_chart.add_image(img, 'A1')
ws_chart.column_dimensions['A'].width = 100

# Сохраняем Excel-файл
output_path = 'prolongation_report.xlsx'
wb.save(output_path)
print(f"Отчёт сохранён в файл: {output_path}")

# Скачивание файла (для Google Colab)
try:
    from google.colab import files
    files.download(output_path)
    # Удаляем временный файл с графиком
    if os.path.exists(chart_path):
        os.remove(chart_path)
except:
    print("Файл сохранён локально")
    if os.path.exists(chart_path):
        os.remove(chart_path)

Отчёт сохранён в файл: prolongation_report.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>